### Import


In [4]:
from pyspark.sql import SparkSession

### Create SparkSession

In [5]:
spark = (
    SparkSession.builder
    .appName("Retail Iceberg")
    .config(
        "spark.sql.catalog.retail",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.retail.type",
        "hadoop"
    )
    .config(
        "spark.sql.catalog.retail.warehouse",
        "/home/iceberg/warehouse"
    )
    .config(
        "spark.sql.defaultCatalog",
        "retail"
    )
    .getOrCreate()
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/07 07:17:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/07 07:17:34 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [6]:
spark 


In [7]:
print(spark.version)

3.5.5


### Testing id the Iceberg catalog exists

In [17]:
spark.sql("show catalogs").show()

+-------------+
|      catalog|
+-------------+
|       retail|
|spark_catalog|
+-------------+



In [8]:
spark.conf.get("spark.sql.catalog.retail", "NOT SET")

'org.apache.iceberg.spark.SparkCatalog'

In [9]:
spark.conf.get("spark.sql.catalog.retail.type", "NOT SET")

'hadoop'

In [10]:
spark.conf.get("spark.sql.catalog.retail.uri", "NOT SET")

'NOT SET'

In [11]:
spark.conf.get("spark.sql.defaultCatalog", "NOT SET")

'demo'

In [12]:
spark.conf.get("spark.sql.catalog.demo")

'org.apache.iceberg.spark.SparkCatalog'

In [13]:
spark.conf.get("spark.sql.catalog.demo.uri")

'http://rest:8181'

### Creating the first DB

In [21]:
spark.sql("CREATE DATABASE IF NOT EXISTS retail.sales_db") #retail=catalog and sales_db=database

DataFrame[]

In [22]:
spark.sql("SHOW DATABASES IN retail").show()

+---------+
|namespace|
+---------+
| sales_db|
+---------+



### A table

In [23]:
spark.sql("""
    CREATE TABLE retail.sales_db.sales(
        transaction_id BIGINT,
        product STRING,
        category STRING,
        quantity INT,
        price DOUBLE,
        sales_date DATE
    )
    USING iceberg
"""
)

DataFrame[]

In [24]:
spark.sql("SHOW TABLES IN retail.sales_db").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
| sales_db|    sales|      false|
+---------+---------+-----------+



### Inserting data

In [26]:
spark.sql("""
    INSERT INTO retail.sales_db.sales VALUES
    (1,'Laptop','Electronics',1,1200,DATE '2026-08-01'),
    (2,'Mouse','Electronics',2,25,DATE '2026-08-01'),
    (3,'Keyboard','Electronics',1,75,DATE '2026-08-02'),
    (4,'Chair','Furniture',1,180,DATE '2026-08-02'),
    (5,'Desk','Furniture',1,350,DATE '2026-08-03')
""")

DataFrame[]

In [27]:
spark.sql("SELECT * FROM retail.sales_db.sales").show()

+--------------+--------+-----------+--------+------+----------+
|transaction_id| product|   category|quantity| price|sales_date|
+--------------+--------+-----------+--------+------+----------+
|             1|  Laptop|Electronics|       1|1200.0|2026-08-01|
|             2|   Mouse|Electronics|       2|  25.0|2026-08-01|
|             3|Keyboard|Electronics|       1|  75.0|2026-08-02|
|             4|   Chair|  Furniture|       1| 180.0|2026-08-02|
|             5|    Desk|  Furniture|       1| 350.0|2026-08-03|
+--------------+--------+-----------+--------+------+----------+



### Time Table

#### Create new snapshot

In [28]:
spark.sql("""
    INSERT INTO retail.sales_db.sales VALUES
    (6,'Monitor','Electronics',1,300,DATE '2026-08-04'),
    (7,'Headphones','Electronics',1,150,DATE '2026-08-04'),
    (8,'Table','Furniture',1,400,DATE '2026-08-04')
""")

DataFrame[]

In [29]:
spark.sql("""
    SELECT * 
    FROM retail.sales_db.sales
    ORDER BY transaction_id
""").show()

+--------------+----------+-----------+--------+------+----------+
|transaction_id|   product|   category|quantity| price|sales_date|
+--------------+----------+-----------+--------+------+----------+
|             1|    Laptop|Electronics|       1|1200.0|2026-08-01|
|             2|     Mouse|Electronics|       2|  25.0|2026-08-01|
|             3|  Keyboard|Electronics|       1|  75.0|2026-08-02|
|             4|     Chair|  Furniture|       1| 180.0|2026-08-02|
|             5|      Desk|  Furniture|       1| 350.0|2026-08-03|
|             6|   Monitor|Electronics|       1| 300.0|2026-08-04|
|             7|Headphones|Electronics|       1| 150.0|2026-08-04|
|             8|     Table|  Furniture|       1| 400.0|2026-08-04|
+--------------+----------+-----------+--------+------+----------+



### Checking snapshots

In [9]:
spark.sql("""
SELECT *
FROM retail.sales_db.sales
ORDER BY transaction_id
""").show()

+--------------+----------+-----------+--------+------+----------+
|transaction_id|   product|   category|quantity| price|sales_date|
+--------------+----------+-----------+--------+------+----------+
|             1|    Laptop|Electronics|       1|1200.0|2026-08-01|
|             2|     Mouse|Electronics|       2|  25.0|2026-08-01|
|             3|  Keyboard|Electronics|       1|  75.0|2026-08-02|
|             4|     Chair|  Furniture|       1| 180.0|2026-08-02|
|             5|      Desk|  Furniture|       1| 350.0|2026-08-03|
|             6|   Monitor|Electronics|       1| 300.0|2026-08-04|
|             7|Headphones|Electronics|       1| 150.0|2026-08-04|
|             8|     Table|  Furniture|       1| 400.0|2026-08-04|
+--------------+----------+-----------+--------+------+----------+



In [8]:
spark.sql("""
    SELECT *
    FROM retail.sales_db.sales.snapshots
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                       |summary                                                                                                                                                     

In [10]:
snapshots = spark.sql("""
    SELECT snapshot_id,parent_id,committed_At,operation
    FROM retail.sales_db.sales.snapshots
    ORDER BY committed_at

""")

In [11]:
snapshots.show(truncate=False)

+-------------------+-------------------+-----------------------+---------+
|snapshot_id        |parent_id          |committed_At           |operation|
+-------------------+-------------------+-----------------------+---------+
|1350242478421060038|NULL               |2026-08-06 09:22:33.619|append   |
|713981930178453912 |1350242478421060038|2026-08-06 15:56:10.307|append   |
+-------------------+-------------------+-----------------------+---------+



### Using Time Travel

In [13]:
spark.sql("""
    SELECT *
    FROM retail.sales_db.sales
    VERSION AS OF 1350242478421060038
    ORDER BY transaction_id
""").show()

+--------------+--------+-----------+--------+------+----------+
|transaction_id| product|   category|quantity| price|sales_date|
+--------------+--------+-----------+--------+------+----------+
|             1|  Laptop|Electronics|       1|1200.0|2026-08-01|
|             2|   Mouse|Electronics|       2|  25.0|2026-08-01|
|             3|Keyboard|Electronics|       1|  75.0|2026-08-02|
|             4|   Chair|  Furniture|       1| 180.0|2026-08-02|
|             5|    Desk|  Furniture|       1| 350.0|2026-08-03|
+--------------+--------+-----------+--------+------+----------+



### Timetravel using committed time

In [14]:
spark.sql("""
    SELECT *
    FROM retail.sales_db.sales
    TIMESTAMP AS OF '2026-08-06 09:22:33.619'
    ORDER BY transaction_id
""").show()

+--------------+--------+-----------+--------+------+----------+
|transaction_id| product|   category|quantity| price|sales_date|
+--------------+--------+-----------+--------+------+----------+
|             1|  Laptop|Electronics|       1|1200.0|2026-08-01|
|             2|   Mouse|Electronics|       2|  25.0|2026-08-01|
|             3|Keyboard|Electronics|       1|  75.0|2026-08-02|
|             4|   Chair|  Furniture|       1| 180.0|2026-08-02|
|             5|    Desk|  Furniture|       1| 350.0|2026-08-03|
+--------------+--------+-----------+--------+------+----------+

